# 06. TabNet 딥러닝 (Attention-based Tabular)

트리 모델 외에 **TabNet**(어텐션 기반 테이블 딥러닝)으로 폐업 위험 분류를 시도한 기록이다.
두 가지 접근을 정리한다.

1. **기본 + optuna 튜닝**: 피처 엔지니어링 후 TabNet을 optuna로 튜닝하고 평가·저장한다.
2. **누수 제거(leakage-free)**: 현재 분기 데이터를 전부 배제하고 과거(t-1, t-2) 정보만으로 **다음 분기** 위험을 예측한다.

> 원본은 Colab/Mac 환경(`google.colab`, MPS)이었다. 여기서는 로컬 실행에 맞게 경로·디바이스를 정리하고 optuna 탐색 횟수와 epoch을 축소했다. 구조는 동일하다.

## 0. 공통 설정

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import matplotlib.font_manager as fm
import matplotlib
font_path = 'C:\\Windows\\Fonts\\gulim.ttc'
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc('font', family=font)
plt.rcParams['axes.unicode_minus'] = False

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             precision_recall_curve, roc_curve, average_precision_score)
from pytorch_tabnet.tab_model import TabNetClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# 디바이스 자동 감지 (로컬: CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

DATA_PATH = '../data/eda/merged_data.csv'
N_TRIALS = 5      # 원본 30에서 축소 (CPU 실행 고려)
TUNE_EPOCHS = 30  # 튜닝 중 epoch
FINAL_EPOCHS = 60 # 최종 학습 epoch

## 1. TabNet 기본 + Optuna 튜닝

상위 50%를 High Risk로 라벨링하고, 매출/인구/경쟁 관련 파생변수를 만든 뒤 TabNet을 optuna로 튜닝한다. (피처 엔지니어링은 당분기 데이터 기반)

In [ ]:
df = pd.read_csv(DATA_PATH)
print('데이터 Shape:', df.shape)

# 폐업률 상위 50% → High Risk(1)
threshold = df['폐업_률'].quantile(0.5)
df['closure_risk'] = (df['폐업_률'] >= threshold).astype(int)
print('임계값:', round(threshold, 2), '| 클래스 분포:', df['closure_risk'].value_counts().to_dict())

In [ ]:
# 범주형 인코딩
le_district = LabelEncoder(); df['자치구_encoded'] = le_district.fit_transform(df['자치구_코드_명'])
le_industry = LabelEncoder(); df['업종_encoded'] = le_industry.fit_transform(df['서비스_업종_코드_명'])
le_change = LabelEncoder(); df['상권변화_encoded'] = le_change.fit_transform(df['상권_변화_지표'])

df['year'] = df['기준_년분기_코드'].astype(str).str[:4].astype(int)
df['quarter'] = df['기준_년분기_코드'].astype(str).str[4:].astype(int)
df['year_quarter'] = df['year'] * 10 + df['quarter']

# Feature Engineering
df = df.sort_values(['자치구_encoded', '업종_encoded', 'year_quarter'])
df['점포당_매출'] = df['당월_매출_금액'] / (df['점포_수'] + 1)
df['건당_매출'] = df['당월_매출_금액'] / (df['당월_매출_건수'] + 1)
df['점포당_유동인구'] = df['총_유동인구_수'] / (df['점포_수'] + 1)
df['개업_률_lag1'] = df.groupby(['자치구_encoded', '업종_encoded'])['개업_률'].shift(1)
df['폐업_률_lag1'] = df.groupby(['자치구_encoded', '업종_encoded'])['폐업_률'].shift(1)
df['개폐업_비율_lag1'] = df['개업_률_lag1'] / (df['폐업_률_lag1'] + 0.1)
df['남성_매출_비율'] = df['남성_매출_금액'] / (df['당월_매출_금액'] + 1)
df['여성_매출_비율'] = df['여성_매출_금액'] / (df['당월_매출_금액'] + 1)
df['주말_매출_비율'] = (df['토요일_매출_금액'] + df['일요일_매출_금액']) / (df['당월_매출_금액'] + 1)
df['야간_매출_비율'] = (df['시간대_00_06_매출_금액'] + df['시간대_21_24_매출_금액']) / (df['당월_매출_금액'] + 1)
df['총_인구'] = df['총_유동인구_수'] + df['총_상주인구_수'] + df['총_직장인구_수']
df['1인당_소득'] = df['월_평균_소득_금액'] / (df['총_상주인구_수'] + 1)
df['소득지출비율'] = df['월_평균_소득_금액'] / (df['지출_총_금액'] + 1)
df['유동인구_밀집도'] = df['총_유동인구_수'] / (df['총_상주인구_수'] + 1)
df['직장인구_비율'] = df['총_직장인구_수'] / (df['총_상주인구_수'] + 1)
df['프랜차이즈_비율'] = df['프랜차이즈_점포_수'] / (df['점포_수'] + 1)
df['경쟁도'] = df['유사_업종_점포_수'] / (df['점포_수'] + 1)
print('피처 엔지니어링 후 컬럼 수:', len(df.columns))

In [ ]:
# Feature 선택
categorical_features = ['자치구_encoded', '업종_encoded', '상권변화_encoded']
exclude_cols = [
    '기준_년분기_코드', '자치구_코드_명', '서비스_업종_코드_명', '상권_변화_지표',
    '폐업_률', '폐업_점포_수', 'closure_risk',
    'year', 'quarter', 'year_quarter',
]
feature_cols = [c for c in df.columns if c not in exclude_cols]
cat_idxs = [feature_cols.index(c) for c in categorical_features]
cat_dims = [len(le_district.classes_), len(le_industry.classes_), len(le_change.classes_)]

# 결측/무한대 처리
df = df.replace([np.inf, -np.inf], np.nan)
df_clean = df.dropna(subset=feature_cols)
print('사용 feature 수:', len(feature_cols), '| 정리 후 shape:', df_clean.shape)

In [ ]:
# 시계열 분할 (train 0.7 / val 0.1 / test 0.2)
df_sorted = df_clean.sort_values('year_quarter')
n = len(df_sorted); n_tr = int(n * 0.7); n_val = int(n * 0.1)
train_df, val_df, test_df = df_sorted.iloc[:n_tr], df_sorted.iloc[n_tr:n_tr+n_val], df_sorted.iloc[n_tr+n_val:]

cat_mask = np.zeros(len(feature_cols), dtype=bool)
for idx in cat_idxs:
    cat_mask[idx] = True
cont_mask = ~cat_mask

X_train = train_df[feature_cols].values; y_train = train_df['closure_risk'].values
X_val = val_df[feature_cols].values; y_val = val_df['closure_risk'].values
X_test = test_df[feature_cols].values; y_test = test_df['closure_risk'].values
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

In [ ]:
# 연속형만 스케일링
scaler = StandardScaler()
X_train_c = scaler.fit_transform(X_train[:, cont_mask].astype(np.float64))
X_val_c = scaler.transform(X_val[:, cont_mask].astype(np.float64))
X_test_c = scaler.transform(X_test[:, cont_mask].astype(np.float64))

X_train_s = X_train.copy().astype(np.float64); X_train_s[:, cont_mask] = X_train_c
X_val_s = X_val.copy().astype(np.float64); X_val_s[:, cont_mask] = X_val_c
X_test_s = X_test.copy().astype(np.float64); X_test_s[:, cont_mask] = X_test_c

X_train_s = X_train_s.astype(np.float32)
X_val_s = X_val_s.astype(np.float32)
X_test_s = X_test_s.astype(np.float32)

In [ ]:
# Optuna 튜닝 (trials/epoch 축소)
def objective(trial):
    params = {
        'mask_type': trial.suggest_categorical('mask_type', ['entmax', 'sparsemax']),
        'n_d': trial.suggest_int('n_d', 8, 32, step=8),
        'n_steps': trial.suggest_int('n_steps', 3, 6),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0, step=0.1),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-6, 1e-3, log=True),
        'lr': trial.suggest_float('lr', 1e-3, 3e-2, log=True),
    }
    model = TabNetClassifier(
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=3,
        n_d=params['n_d'], n_a=params['n_d'], n_steps=params['n_steps'],
        gamma=params['gamma'], lambda_sparse=params['lambda_sparse'],
        optimizer_params=dict(lr=params['lr']),
        scheduler_params={'step_size': 50, 'gamma': 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type=params['mask_type'], seed=SEED, verbose=0, device_name=device)
    model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)],
              max_epochs=TUNE_EPOCHS, patience=8, eval_metric=['auc'])
    return roc_auc_score(y_val, model.predict_proba(X_val_s)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS)
best_params = study.best_trial.params
print('Best val AUC:', round(study.best_value, 4))
print('Best params:', best_params)

In [ ]:
# 최적 파라미터로 최종 학습
model = TabNetClassifier(
    cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=3,
    n_d=best_params['n_d'], n_a=best_params['n_d'], n_steps=best_params['n_steps'],
    gamma=best_params['gamma'], lambda_sparse=best_params['lambda_sparse'],
    optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=best_params['lr']),
    scheduler_params={'step_size': 50, 'gamma': 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type=best_params['mask_type'], seed=SEED, verbose=10, device_name=device)

model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)],
          eval_metric=['accuracy', 'auc', 'logloss'],
          max_epochs=FINAL_EPOCHS, patience=15, batch_size=1024, virtual_batch_size=256)
print('Best epoch:', model.best_epoch)

In [ ]:
# 임계값 최적화(F1) 후 테스트 평가
y_val_proba = model.predict_proba(X_val_s)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, y_val_proba)
f1s = 2 * (prec * rec) / (prec + rec + 1e-10)
opt_idx = int(np.argmax(f1s))
opt_thr = thr[opt_idx] if opt_idx < len(thr) else 0.5

y_test_proba = model.predict_proba(X_test_s)[:, 1]
y_test_pred = (y_test_proba >= opt_thr).astype(int)

print(f'최적 임계값: {opt_thr:.4f}')
print(f'Accuracy : {accuracy_score(y_test, y_test_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_test_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_test_pred):.4f}')
print(f'F1       : {f1_score(y_test, y_test_pred):.4f}')
print(f'AUC      : {roc_auc_score(y_test, y_test_proba):.4f}')
print(classification_report(y_test, y_test_pred, target_names=['Low Risk', 'High Risk']))

In [ ]:
# 혼동행렬 / ROC / PR
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'], ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual'); axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[1].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_test_proba):.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].set_title('ROC Curve'); axes[1].legend()

pr, rc, _ = precision_recall_curve(y_test, y_test_proba)
axes[2].plot(rc, pr, label=f'AP = {average_precision_score(y_test, y_test_proba):.3f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision'); axes[2].set_title('PR Curve'); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Feature Importance (Top 20)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df, x='importance', y='feature')
plt.title('TabNet Feature Importance (Top 20)')
plt.tight_layout(); plt.show()

## 2. 누수 제거 버전 (Leakage-free, 미래 예측)

**현재 분기(t)의 모든 원본 데이터를 배제**하고, 과거(t-1, t-2) lag 특성만으로 다음 시점의 폐업 위험을 예측한다. 그룹 통계량도 train에서만 계산해 데이터 누수를 차단한다.

> 폐업_률·폐업_점포_수는 피처로 쓰지 않는다 (타깃 누수 방지).

In [ ]:
df = pd.read_csv(DATA_PATH)
print('데이터 Shape:', df.shape)

df['year'] = df['기준_년분기_코드'].astype(str).str[:4].astype(int)
df['quarter'] = df['기준_년분기_코드'].astype(str).str[4:].astype(int)
df['year_quarter'] = df['year'] * 10 + df['quarter']

le_district = LabelEncoder(); df['자치구_encoded'] = le_district.fit_transform(df['자치구_코드_명'])
le_industry = LabelEncoder(); df['업종_encoded'] = le_industry.fit_transform(df['서비스_업종_코드_명'])
le_change = LabelEncoder(); df['상권변화_encoded'] = le_change.fit_transform(df['상권_변화_지표'])

df = df.sort_values(['자치구_encoded', '업종_encoded', 'year_quarter']).reset_index(drop=True)

# 타깃: 폐업률 중앙값 기준 High Risk
threshold = df['폐업_률'].quantile(0.5)
df['closure_risk'] = (df['폐업_률'] >= threshold).astype(int)
df = df.dropna(subset=['폐업_률']).reset_index(drop=True)
print('임계값:', round(threshold, 2), '| 분포:', df['closure_risk'].value_counts().to_dict())

In [ ]:
# 과거(t-1, t-2) lag 특성만 생성 (현재 시점 금지)
lag_cols = [
    '당월_매출_금액', '당월_매출_건수', '점포_수', '유사_업종_점포_수',
    '프랜차이즈_점포_수', '개업_률', '개업_점포_수',
    '전체임대료', '총_유동인구_수', '총_상주인구_수', '총_직장인구_수',
    '토요일_매출_금액', '일요일_매출_금액', '시간대_21_24_매출_금액',
    '연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액',
    '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액',
]
for col in lag_cols:
    df[f'{col}_lag1'] = df.groupby(['자치구_encoded', '업종_encoded'])[col].shift(1)
    df[f'{col}_lag2'] = df.groupby(['자치구_encoded', '업종_encoded'])[col].shift(2)

# 과거 기반 파생
df['매출_변화율'] = (df['당월_매출_금액_lag1'] - df['당월_매출_금액_lag2']) / (df['당월_매출_금액_lag2'] + 1)
df['점포수_변화율'] = (df['점포_수_lag1'] - df['점포_수_lag2']) / (df['점포_수_lag2'] + 1)
df['점포당_매출'] = df['당월_매출_금액_lag1'] / (df['점포_수_lag1'] + 1)
df['임대료_부담률'] = df['전체임대료_lag1'] / (df['당월_매출_금액_lag1'] + 1)
df['프랜차이즈_비율'] = df['프랜차이즈_점포_수_lag1'] / (df['점포_수_lag1'] + 1)
df['경쟁_밀도'] = df['유사_업종_점포_수_lag1'] / (df['점포_수_lag1'] + 1)

df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
print('lag 특성 생성 완료, 컬럼 수:', len(df.columns))

In [ ]:
# Feature 선택: 현재 시점(t) 원본 전부 제외, lag/파생/범주형만
categorical_features = ['자치구_encoded', '업종_encoded', '상권변화_encoded']
derived = ['매출_변화율', '점포수_변화율', '점포당_매출', '임대료_부담률', '프랜차이즈_비율', '경쟁_밀도']
lag_feats = [c for c in df.columns if c.endswith('_lag1') or c.endswith('_lag2')]
feature_cols = categorical_features + lag_feats + derived

cat_idxs = [feature_cols.index(c) for c in categorical_features]
cat_dims = [len(le_district.classes_), len(le_industry.classes_), len(le_change.classes_)]
print('사용 feature 수:', len(feature_cols))

In [ ]:
# 각 그룹 초기 2개 분기 제거(lag2 유효화) 후 시계열 분할
df_clean = df.dropna(subset=feature_cols).reset_index(drop=True)
df_clean = df_clean.groupby(['자치구_encoded', '업종_encoded'], group_keys=False).apply(
    lambda x: x.iloc[2:] if len(x) > 2 else x).reset_index(drop=True)
df_sorted = df_clean.sort_values(['year_quarter', '자치구_encoded', '업종_encoded']).reset_index(drop=True)

n = len(df_sorted); n_tr = int(n * 0.7); n_val = int(n * 0.1)
train_df, val_df, test_df = df_sorted.iloc[:n_tr], df_sorted.iloc[n_tr:n_tr+n_val], df_sorted.iloc[n_tr+n_val:]
print('Train/Val/Test:', len(train_df), len(val_df), len(test_df))

In [ ]:
# 그룹 통계량은 train에서만 계산 (누수 방지)
district_mean = train_df.groupby('자치구_encoded')['당월_매출_금액_lag1'].mean()
industry_mean = train_df.groupby('업종_encoded')['당월_매출_금액_lag1'].mean()
for d in [train_df, val_df, test_df]:
    d['상권_매출_평균'] = d['자치구_encoded'].map(district_mean)
    d['업종_매출_평균'] = d['업종_encoded'].map(industry_mean)
    d['상권_대비_매출'] = d['당월_매출_금액_lag1'] / (d['상권_매출_평균'] + 1)
    d['업종_대비_매출'] = d['당월_매출_금액_lag1'] / (d['업종_매출_평균'] + 1)

feature_cols = feature_cols + ['상권_매출_평균', '업종_매출_평균', '상권_대비_매출', '업종_대비_매출']
cat_idxs = [feature_cols.index(c) for c in categorical_features]

for d in [train_df, val_df, test_df]:
    d[feature_cols] = d[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
cat_mask = np.zeros(len(feature_cols), dtype=bool)
for idx in cat_idxs:
    cat_mask[idx] = True
cont_mask = ~cat_mask

X_train = train_df[feature_cols].values; y_train = train_df['closure_risk'].values
X_val = val_df[feature_cols].values; y_val = val_df['closure_risk'].values
X_test = test_df[feature_cols].values; y_test = test_df['closure_risk'].values

scaler = StandardScaler()
Xtr_c = scaler.fit_transform(X_train[:, cont_mask].astype(np.float64))
Xva_c = scaler.transform(X_val[:, cont_mask].astype(np.float64))
Xte_c = scaler.transform(X_test[:, cont_mask].astype(np.float64))

X_train_s = X_train.copy().astype(np.float64); X_train_s[:, cont_mask] = Xtr_c
X_val_s = X_val.copy().astype(np.float64); X_val_s[:, cont_mask] = Xva_c
X_test_s = X_test.copy().astype(np.float64); X_test_s[:, cont_mask] = Xte_c
X_train_s = X_train_s.astype(np.float32); X_val_s = X_val_s.astype(np.float32); X_test_s = X_test_s.astype(np.float32)
print('준비 완료:', X_train_s.shape, X_val_s.shape, X_test_s.shape)

In [ ]:
# 누수 제거 TabNet 학습 (고정 파라미터)
model = TabNetClassifier(
    cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=3,
    n_d=8, n_a=8, n_steps=3, gamma=1.4, lambda_sparse=9e-6,
    optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=0.01),
    scheduler_params={'step_size': 50, 'gamma': 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax', seed=SEED, verbose=10, device_name=device)

model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)],
          eval_metric=['accuracy', 'auc', 'logloss'],
          max_epochs=FINAL_EPOCHS, patience=15, batch_size=1024, virtual_batch_size=128)
print('Best epoch:', model.best_epoch)

In [ ]:
# 평가 (ROC 기반 최적 임계값)
y_val_proba = model.predict_proba(X_val_s)[:, 1]
fpr, tpr, roc_thr = roc_curve(y_val, y_val_proba)
opt_thr = roc_thr[int(np.argmax(tpr - fpr))]

y_test_proba = model.predict_proba(X_test_s)[:, 1]
y_test_pred = (y_test_proba >= opt_thr).astype(int)

print(f'최적 임계값: {opt_thr:.4f}')
print(f'Accuracy : {accuracy_score(y_test, y_test_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_test_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_test_pred):.4f}')
print(f'F1       : {f1_score(y_test, y_test_pred):.4f}')
print(f'AUC      : {roc_auc_score(y_test, y_test_proba):.4f}')
print(classification_report(y_test, y_test_pred, target_names=['Low Risk', 'High Risk']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'], ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual'); axes[0].set_title('Leakage-free Confusion Matrix')

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(15)
sns.barplot(data=importance_df, x='importance', y='feature', ax=axes[1])
axes[1].set_title('Feature Importance (Top 15)')
plt.tight_layout(); plt.show()

## 정리

- **기본 + optuna**: 당분기 피처 + 파생변수로 TabNet을 튜닝. 트리 모델과 비슷한 수준의 분류 성능.
- **누수 제거**: 현재 시점 데이터를 모두 배제하고 과거 정보만 쓰면 성능은 다소 떨어지지만, **실제 운영에서 쓸 수 있는 정직한 예측력**을 보여준다.

딥러닝(TabNet)은 트리 부스팅 대비 뚜렷한 우위가 없었고, 최종 서비스 모델로는 해석과 속도가 유리한 **CatBoost(05)** 를 채택했다.